## Ask: Context

Dunder Mifflin Paper Company is rebuilding how its sales organisation forecasts.
The CRO wants three questions answered from the CRM export:

1. We missed Q1 bookings by 18%. What blew the forecast?
2. Where are deals stalling, and why?
3. Given today's pipe, are we on track for next quarter?

The brief is deliberately ambiguous. State assumptions and caveats explicitly.


## **Prepare** : Setup with Imports and Ingestion

### Imports and Ingestion

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

DATA = Path('../data')


In [ ]:
def load(tab: str) -> pd.DataFrame:
    """Read a dataset tab from CSV.

    Everything is read as string so the cleaning layer owns all type coercion.
    This mirrors the original Sheets ingestion (numericise_ignore=['all']) and
    keeps the Process section as the single place where types are decided.
    """
    df = (pd.read_csv(DATA / f'{tab}.csv', dtype=str)
            .replace('', pd.NA)
            .dropna(how='all')
            .reset_index(drop=True))
    print(f'  {tab:16s} -> {df.shape[0]:>5} rows x {df.shape[1]:>2} cols')
    return df


deals_meta     = load('deals_meta')
deals_snapshot = load('deals_snapshot')
owner          = load('owners')
targets        = load('targets')


## **Process** : Cleaning, Assumptions, Normalization and Modelling derived tables

### Cleaning

In [ ]:
# helpers
def safe_date(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, infer_datetime_format=True, errors='coerce')

def safe_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(
        s.astype(str).str.replace(r'[\$,\s]', '', regex=True),
        errors='coerce'
    )

def to_quarter(s: pd.Series) -> pd.Series:
    return (s.dt.to_period('Q')
             .astype(str)
             .str.replace(r'(\d{4})Q(\d)', r'\1 Q\2', regex=True))

def split_quarter(quarter_series: pd.Series):
    year = quarter_series.str[:4].astype('Int64')
    qnum = quarter_series.str[-1].astype('Int64')
    return year, qnum


In [ ]:
#stage constants
STAGE_ORDER = {
    'Prospecting'  : 1,
    'Qualification': 2,
    'Proposal'     : 3,
    'Negotiation'  : 4,
    'Closed Won'   : 5,
    'Closed Lost'  : 6,}

STAGE_LABEL     = {v: k for k, v in STAGE_ORDER.items()}
OPEN_STAGES     = [1, 2, 3, 4]

QUARTERS_COMPLETOS = ['2024 Q1','2024 Q2','2024 Q3','2024 Q4','2025 Q1','2025 Q2','2025 Q3',]

### Assumptions

In [ ]:
# temporal references + thresholds
STALL_WEEKS = 4
Q1_ANALYSIS = '2025 Q1'
NEXT_Q      = '2025 Q3'

In [ ]:
# forecast weights placeholder
FORECAST_WEIGHTS = {
    'Commit'   : 0.90,
    'Best Case': 0.50,
    'Pipeline' : 0.20,
}

In [ ]:
# segment growth rates
SEGMENT_GROWTH = {
    'SMB'        : 0.15,
    'Mid-Market' : 0.12,
    'Enterprise' : 0.10,
    'Strategic'  : 0.08,
}

In [ ]:
# quarter sequence stabilizer
QUARTER_SEQUENCE = [
    '2024 Q1','2024 Q2','2024 Q3','2024 Q4',
    '2025 Q1','2025 Q2','2025 Q3','2025 Q4',
]
QUARTER_INDEX = {q: i for i, q in enumerate(QUARTER_SEQUENCE)}

### Normalization

In [ ]:
# deals meta data normalized
deals_meta['created_date'] = safe_date(deals_meta['created_date'])
deals_meta['close_date']   = safe_date(deals_meta['close_date'])
deals_meta['deal_amount']  = safe_num(deals_meta['deal_amount'])
deals_meta['deal_stage']   = deals_meta['deal_stage'].astype(str).str.strip()

deals_meta['close_quarter']   = to_quarter(deals_meta['close_date'])
deals_meta['close_month']     = deals_meta['close_date'].dt.to_period('M').astype(str)
deals_meta['created_quarter'] = to_quarter(deals_meta['created_date'])

deals_meta['close_year'],   deals_meta['close_qnum']   = split_quarter(deals_meta['close_quarter'])
deals_meta['created_year'], deals_meta['created_qnum'] = split_quarter(deals_meta['created_quarter'])

deals_meta['stage_order'] = deals_meta['deal_stage'].map(STAGE_ORDER)
deals_meta['is_won']      = deals_meta['stage_order'] == 5
deals_meta['is_lost']     = deals_meta['stage_order'] == 6
deals_meta['is_open']     = deals_meta['stage_order'].isin(OPEN_STAGES)
deals_meta['is_closed']   = deals_meta['is_won'] | deals_meta['is_lost']

In [ ]:
# deals snapshot data normalized
deals_snapshot['snapshot_date'] = safe_date(deals_snapshot['snapshot_date'])
deals_snapshot['close_date']    = safe_date(deals_snapshot['close_date'])
deals_snapshot['amount']        = safe_num(deals_snapshot['amount'])
deals_snapshot['stage']         = deals_snapshot['stage'].astype(str).str.strip()

deals_snapshot['snap_quarter'] = to_quarter(deals_snapshot['snapshot_date'])
deals_snapshot['snap_month']   = deals_snapshot['snapshot_date'].dt.to_period('M').astype(str)
deals_snapshot['snap_year'], deals_snapshot['snap_qnum'] = split_quarter(deals_snapshot['snap_quarter'])

deals_snapshot['stage_order']  = deals_snapshot['stage'].map(STAGE_ORDER)

In [ ]:
# owners data normalized
owner['role_start_date'] = safe_date(owner['role_start_date'])
owner['role_end_date']   = safe_date(owner['role_end_date'])

owner['is_active']       = owner['role_end_date'].isna()

In [ ]:
# targets data normalized
targets['target_amount']                = safe_num(targets['target_amount'])
targets                                 = targets.drop(columns=['segment'], errors='ignore')
targets['year'], targets['quarter_num'] = split_quarter(targets['quarter'])
targets['target_amount_raw']            = targets['target_amount'].copy()

owner_segment_map = owner.set_index('owner_id')['segment'].to_dict()
BASE_NORM_Q = '2024 Q1'   # normalization base, always the only calibrated quarter
base_q1     = (targets[targets['quarter'] == BASE_NORM_Q]
                .set_index('owner_id')['target_amount']
                .to_dict())

targets['normalized_target'] = targets.apply(
    lambda row: round(
        base_q1.get(row['owner_id'], np.nan) *
        (1 + SEGMENT_GROWTH.get(
            owner_segment_map.get(row['owner_id']), 0.12)
        ) ** QUARTER_INDEX.get(row['quarter'], 0),
        0
    ),
    axis=1
)
targets['target_amount'] = targets['normalized_target']

### Sanity Check

In [ ]:
# quarters format
print('deals_meta   close_quarter:', sorted(deals_meta['close_quarter'].dropna().unique()))
print('deals_snap   snap_quarter: ', sorted(deals_snapshot['snap_quarter'].dropna().unique()))
print('targets      quarter:      ', sorted(targets['quarter'].unique()))

In [ ]:
# stage mapping corretion
for label, col, df in [('deals_meta',     'deal_stage', deals_meta),
                        ('deals_snapshot', 'stage',      deals_snapshot)]:
    unknown = df[df['stage_order'].isna()][col].value_counts()
    if not unknown.empty:
        print(f'\n Stages not recognized on {label}:')
        print(unknown.to_string())
    else:
        print(f'{label}: all stages mapped')


In [ ]:
# nulls
for name, df in [('deals_meta',     deals_meta),
                 ('deals_snapshot', deals_snapshot),
                 ('owner',          owner),
                 ('targets',        targets)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if not nulls.empty:
        if name == 'owner' and set(nulls.index) == {'role_end_date'}:
            print(f'ℹ️  owner.role_end_date: {nulls[0]} nulls = active owners OK')
        else:
            print(f'\ nulls in {name}:')
            print(nulls.to_string())

In [ ]:
# normalized target varidation
print('\n Raw vs Normalized target — owner_1 (SMB)')
print(targets[targets['owner_id'] == 'owner_1']
      [['quarter','target_amount_raw','normalized_target']].to_string())

print('\n Total raw vs normalized por quarter')
comp = (targets.groupby('quarter')
        .agg(raw=('target_amount_raw','sum'),
             normalized=('normalized_target','sum'))
        .reset_index()
        .sort_values('quarter'))
comp['raw_multiplier'] = (comp['raw'] / comp['normalized']).round(2)
print(comp.to_string())

print('\n Silver layer done')

### Derived Tables

In [ ]:
# deals and owners
deals_enriched = deals_meta.merge(
    owner[['owner_id','name','role','segment','manager','is_active']],
    on='owner_id', how='left'
).rename(columns={
    'name'     : 'owner_name',
    'role'     : 'owner_role',
    'segment'  : 'owner_segment',
    'manager'  : 'owner_manager',
    'is_active': 'owner_is_active',
})

won = deals_enriched[deals_enriched['is_won']].copy()
won['cycle_days'] = (won['close_date'] - won['created_date']).dt.days

In [ ]:
# snapshot base
last_snap = (deals_snapshot
             .groupby('deal_id')['snapshot_date']
             .max()
             .reset_index()
             .rename(columns={'snapshot_date': 'last_snap'}))

In [ ]:
# forecast calibration
any_snap = (deals_snapshot[deals_snapshot['stage_order'].isin(OPEN_STAGES)]
            [['deal_id','forecast_category']]
            .drop_duplicates()
            .merge(deals_meta[['deal_id','is_won','is_closed']],
                   on='deal_id', how='left'))

calibration = (any_snap
               .groupby('forecast_category')
               .agg(
                   deals_visible = ('deal_id',   'nunique'),
                   deals_won      = ('is_won',    'sum'),
                   deals_closed = ('is_closed', 'sum'),
               )
               .assign(
                   pct_won_of_visible = lambda d: (d['deals_won'] / d['deals_visible']).round(3),
                   pct_won_of_closed  = lambda d: (d['deals_won'] / d['deals_closed']).round(3),
               )
               .reset_index())

print('Weight Calibration on Forecast (Category)')
display(calibration)

In [ ]:
# forecast weights
FORECAST_WEIGHTS_CALIBRATED = (calibration
                                .set_index('forecast_category')
                                ['pct_won_of_visible']
                                .to_dict())
FORECAST_WEIGHTS_CALIBRATED.pop('Closed', None)

# original benchmark for comparison and audit
FORECAST_WEIGHTS_BENCHMARK = {
    'Commit'   : 0.90,
    'Best Case': 0.50,
    'Pipeline' : 0.20,
}

# If calibrated Commit < 5%, use benchmark, weekly granularity misses short-duration Commit deals (< 7 days to close)
for cat in ['Commit','Best Case','Pipeline']:
    cal_val = FORECAST_WEIGHTS_CALIBRATED.get(cat, 0)
    bench   = FORECAST_WEIGHTS_BENCHMARK[cat]
    if cal_val < 0.05 and cat == 'Commit':
        FORECAST_WEIGHTS[cat] = bench
        print(f'  {cat:12s}: {cal_val:.1%} → using benchmark {bench:.0%} '
              f'(weekly granularity misses short-duration Commit deals)')
    else:
        FORECAST_WEIGHTS[cat] = cal_val
        print(f'  {cat:12s}: {cal_val:.1%} calibrated from historical data')

print('\n FORECAST_WEIGHTS (final)')
for k, v in sorted(FORECAST_WEIGHTS.items(), key=lambda x: -x[1]):
    print(f'  {k:12s}: {v:.1%}')

In [ ]:
#open pipeline
open_pipeline = (deals_snapshot
                 .merge(last_snap, on='deal_id')
                 .query('snapshot_date == last_snap and stage_order in @OPEN_STAGES')
                 .merge(owner[['owner_id','name','segment','is_active']],
                        on='owner_id', how='left')
                 .rename(columns={
                     'name'     : 'owner_name',
                     'segment'  : 'owner_segment',
                     'is_active': 'owner_is_active',
                 }))

In [ ]:
#stage velocity
ds = (deals_snapshot[deals_snapshot['stage_order'].isin(OPEN_STAGES)]
      .sort_values(['deal_id','snapshot_date'])
      .copy())

ds['prev_stage']    = ds.groupby('deal_id')['stage'].shift(1)
ds['stage_changed'] = (ds['stage'] != ds['prev_stage']) | ds['prev_stage'].isna()
ds['block']         = ds.groupby('deal_id')['stage_changed'].cumsum()

spans = (ds.groupby(['deal_id','stage','stage_order','block'])
           .agg(
               entered        = ('snapshot_date', 'min'),
               last_seen      = ('snapshot_date', 'max'),
               weeks_in_stage = ('snapshot_date', 'count'),
           )
           .reset_index())

velocity = (spans.groupby(['stage','stage_order'])
            .agg(
                deals        = ('deal_id',         'nunique'),
                avg_weeks    = ('weeks_in_stage',   'mean'),
                median_weeks = ('weeks_in_stage',   'median'),
                p90_weeks    = ('weeks_in_stage',   lambda x: x.quantile(0.9)),
                stall_rate   = ('weeks_in_stage',   lambda x: (x > 1).mean()),
            )
            .round(2)
            .reset_index()
            .sort_values('stage_order'))

In [ ]:
# stalled deals
latest_block = (spans.groupby('deal_id')['last_seen']
                .max().reset_index()
                .rename(columns={'last_seen': 'max_last_seen'}))

stalled = (spans
           .merge(latest_block, on='deal_id')
           .query('last_seen == max_last_seen and weeks_in_stage >= @STALL_WEEKS')
           .merge(open_pipeline[['deal_id','amount','owner_name',
                                  'owner_is_active','close_date',
                                  'snap_quarter','snap_year','snap_qnum']],
                  on='deal_id')
           .sort_values('weeks_in_stage', ascending=False)
           .reset_index(drop=True))

In [ ]:
# slip
Q1_YEAR  = int(Q1_ANALYSIS[:4])
Q1_START = pd.Timestamp(f'{Q1_YEAR}-01-01')
Q1_END   = pd.Timestamp(f'{Q1_YEAR}-03-31')

in_q1_ids = set(
    deals_snapshot[
        deals_snapshot['snapshot_date'].between(Q1_START, Q1_END) &
        deals_snapshot['stage_order'].isin(OPEN_STAGES)
    ]['deal_id']
)

final_state = (deals_snapshot
               .merge(last_snap, on='deal_id')
               .query('snapshot_date == last_snap')
               [['deal_id','stage_order','amount']])

slip_df = final_state[final_state['deal_id'].isin(in_q1_ids)].copy()

slip_summary = pd.DataFrame([{
    'deals_in_q1_pipe'   : len(slip_df),
    'closed_won'         : int((slip_df['stage_order'] == 5).sum()),
    'still_open'         : int(slip_df['stage_order'].isin(OPEN_STAGES).sum()),
    'lost'               : int((slip_df['stage_order'] == 6).sum()),
    'open_amount_at_risk': float(
        slip_df.loc[slip_df['stage_order'].isin(OPEN_STAGES), 'amount'].sum()
    ),
}])

In [ ]:
# sanity check
print(f'deals_enriched : {len(deals_enriched):>5} rows')
print(f'won            : {len(won):>5} Closed Won deals')
print(f'open_pipeline  : {len(open_pipeline):>5} open deals')
print(f'spans          : {len(spans):>5} stage blocks')
print(f'stalled        : {len(stalled):>5} deals stalled >= {STALL_WEEKS} weeks')
print(f'\nStage velocity:')
display(velocity)
print(f'\nSlip {Q1_ANALYSIS}:')
display(slip_summary)
print('\nDerived tables done')

## **Analysis** :  Revenue mix and trends, assumptions, pipeline coverage, owner and deals analysis

###   Analysing Revenue Mix and Trends

In [ ]:
# Win rate, ASP, sales cycle, and bookings composition over time.
# Establishes the baseline of how the business generates revenue and identifies the structural shift from New Client to Existing Business.

won_c = won[won['close_quarter'].isin(QUARTERS_COMPLETOS)].copy()

closed          = deals_meta[deals_meta['is_closed']]
win_rate_global = deals_meta['is_won'].sum() / len(closed)

win_rate_by_type = (closed
                    .groupby('deal_order_type')
                    .agg(won=('is_won','sum'), total=('is_won','count'))
                    .assign(win_rate=lambda d: (d['won'] / d['total']).round(3))
                    ['win_rate'])

asp_cycle = (won_c
             .groupby(['owner_segment','deal_order_type'])
             .agg(
                 deals          = ('deal_id',     'count'),
                 asp            = ('deal_amount', 'mean'),
                 avg_cycle_days = ('cycle_days',  'mean'),
             )
             .round(0)
             .reset_index())

bookings_mix = (won_c
                .groupby(['close_quarter','close_year','close_qnum',
                          'deal_order_type','deal_type'])
                .agg(bookings=('deal_amount','sum'), deal_count=('deal_id','count'))
                .reset_index())

bookings_mix = bookings_mix.sort_values(['deal_order_type','close_year','close_qnum'])
bookings_mix['qoq_pct'] = (bookings_mix
                            .groupby('deal_order_type')['bookings']
                            .pct_change() * 100).round(1)

total_q = bookings_mix.groupby('close_quarter')['bookings'].transform('sum')
bookings_mix['pct_of_quarter'] = (bookings_mix['bookings'] / total_q * 100).round(1)

bookings_mix = (bookings_mix
                .sort_values(['close_year','close_qnum','deal_order_type'])
                .reset_index(drop=True))

quarterly = (deals_meta[deals_meta['close_quarter'].isin(QUARTERS_COMPLETOS)]
             .groupby(['close_quarter','close_year','close_qnum'])
             .apply(lambda g: pd.Series({
                 'won'           : int(g['is_won'].sum()),
                 'lost'          : int(g['is_lost'].sum()),
                 'win_rate_pct'  : round(g['is_won'].sum() / max(g['is_closed'].sum(),1)*100, 1),
                 'avg_asp'       : round(g.loc[g['is_won'],'deal_amount'].mean(), 0),
                 'total_bookings': round(g.loc[g['is_won'],'deal_amount'].sum(), 0),
             }))
             .reset_index()
             .sort_values(['close_year','close_qnum'])
             .reset_index(drop=True))

quarterly['bookings_qoq'] = (quarterly['total_bookings'].pct_change() * 100).round(1)

print(f' Global Win Rate:       {win_rate_global:.1%}')
print(f' Historic TCV total:    ${won_c["deal_amount"].sum():>12,.0f}')
print(f' Average ASP:           ${won_c["deal_amount"].mean():>12,.0f}')
print(f' Average Cycle (days):  {won_c["cycle_days"].mean():>12.0f}')
print(f'\n Win Rate by deal type:')
print(win_rate_by_type.to_string())
print(f'\n Quarterly trend:')
display(quarterly[['close_quarter','won','lost','win_rate_pct',
                   'avg_asp','total_bookings','bookings_qoq']])

### Analysing Q1 2025 Attainment

In [ ]:
# Attainment
Q1 = Q1_ANALYSIS

q1_actuals = (won[won['close_quarter'] == Q1]
              .groupby('owner_id')['deal_amount']
              .sum().reset_index()
              .rename(columns={'deal_amount': 'actual_bookings'}))

q1_att = (targets[targets['quarter'] == Q1]
          .merge(owner[['owner_id','name','is_active','segment']],  # ← adicionar 'segment'
                 on='owner_id', how='left')
          .merge(q1_actuals, on='owner_id', how='left')
          .fillna({'actual_bookings': 0}))

q1_att['attainment_pct'] = (q1_att['actual_bookings'] /
                              q1_att['target_amount'].replace(0, np.nan) * 100).round(1)
q1_att['gap'] = (q1_att['actual_bookings'] - q1_att['target_amount']).round(0)

total_target  = q1_att['target_amount'].sum()
total_actuals = q1_att['actual_bookings'].sum()

print(f' Attainment {Q1} (normalized target):')
print(f'   Target:      ${total_target:>10,.0f}')
print(f'   Actuals:     ${total_actuals:>10,.0f}')
print(f'   Miss:        {(1 - total_actuals/total_target)*100:.1f}%')
display(q1_att[['name','segment','is_active','target_amount',
                'actual_bookings','attainment_pct','gap']]
        .sort_values('attainment_pct'))

### Analysing Q2 Proogress

In [ ]:
# Q2 2025 Progress - is the pipeline building fast enough?

last_snapshot  = deals_snapshot['snapshot_date'].max()
q2_start       = pd.Timestamp('2025-04-01')
q2_end         = pd.Timestamp('2025-06-30')
q2_total_weeks = 13

weeks_elapsed   = ((last_snapshot - q2_start).days / 7)
weeks_remaining = q2_total_weeks - weeks_elapsed
pct_elapsed     = weeks_elapsed / q2_total_weeks

q1_start       = pd.Timestamp('2025-01-01')
q1_last_snap   = deals_snapshot[deals_snapshot['snap_quarter'] == '2025 Q1']['snapshot_date'].max()
q1_weeks_avail = ((q1_last_snap - q1_start).days / 7)

print(f'Where are we in Q2 2025?')
print(f'  Last snapshot:     {last_snapshot.date()}')
print(f'  Weeks elapsed:     {weeks_elapsed:.1f} of {q2_total_weeks}')
print(f'  Weeks remaining:   {weeks_remaining:.1f}')
print(f'  Quarter progress:  {pct_elapsed:.0%}')
print(f'\nQ1 2025 data availability:')
print(f'  Last snapshot:     {q1_last_snap.date()}')
print(f'  Weeks available:   {q1_weeks_avail:.1f} of 13')
print(f'  Complete quarter:  {"Yes" if q1_weeks_avail >= 12 else "No — partial"}')

def pipeline_at_week(quarter_label, quarter_start):
    """Open pipeline value at each week number within the quarter."""
    q_snaps = (deals_snapshot[
        (deals_snapshot['snap_quarter'] == quarter_label) &
        (deals_snapshot['stage_order'].isin(OPEN_STAGES))
    ].copy())
    q_snaps['week_in_q'] = (
        (q_snaps['snapshot_date'] - quarter_start).dt.days / 7
    ).astype(int) + 1
    return (q_snaps.groupby('week_in_q')['amount'].sum().reset_index()
            .rename(columns={'amount': f'pipe_{quarter_label}'}))

q1_by_week = pipeline_at_week('2025 Q1', q1_start)
q2_by_week = pipeline_at_week('2025 Q2', q2_start)

comparison = (q1_by_week
              .merge(q2_by_week, on='week_in_q', how='outer')
              .sort_values('week_in_q'))

print(f'\nPipeline by week-in-quarter (Q1 2025 vs Q2 2025):')
print(comparison.to_string())

In [ ]:
# Pipeline drop in last 4 weeks of Q2 2025 - are we building pipeline fast enough for Q3?

won_q1_partial = won[won['close_quarter'] == '2025 Q1']['deal_amount'].sum()
won_q2_partial = won[won['close_quarter'] == '2025 Q2']['deal_amount'].sum()
target_q2      = targets[targets['quarter'] == '2025 Q2']['target_amount'].sum()
expected_by_now = target_q2 * pct_elapsed
run_rate        = won_q2_partial / pct_elapsed

peak_pipe_q2 = comparison['pipe_2025 Q2'].max()
curr_pipe_q2 = comparison['pipe_2025 Q2'].dropna().iloc[-1]
drop_pct     = (curr_pipe_q2 - peak_pipe_q2) / peak_pipe_q2 * 100

print('Q2 2025 — Full Picture')
print(f'\n  BOOKINGS (on track):')
print(f'  Closed so far:          ${won_q2_partial:>10,.0f}')
print(f'  Expected by now ({pct_elapsed:.0%}):  ${expected_by_now:>10,.0f}')
print(f'  Projected run rate:     ${run_rate:>10,.0f}')
print(f'  Q2 target:              ${target_q2:>10,.0f}')
print(f'  Pace vs target:         +${won_q2_partial - expected_by_now:>9,.0f} ahead')

print(f'\n  PIPELINE (warning):')
print(f'  Peak (week 3):          ${peak_pipe_q2:>10,.0f}')
print(f'  Current (week 10):      ${curr_pipe_q2:>10,.0f}')
print(f'  Drop last 4 weeks:      {drop_pct:.0f}%')
print(f'  Q1 same point (wk 10):  ${comparison["pipe_2025 Q1"].iloc[9]:>10,.0f}')
print(f'  Q2 vs Q1 at wk 10:      {(curr_pipe_q2 / comparison["pipe_2025 Q1"].iloc[9] - 1):.0%}')

### Analysing Pipeline Coverage

In [ ]:
pipe_next = (open_pipeline
             .merge(deals_meta[['deal_id','close_quarter']], on='deal_id', how='left')
             .query('close_quarter == @NEXT_Q'))

target_next   = targets[targets['quarter'] == NEXT_Q]['target_amount'].sum()
pipe_total    = pipe_next['amount'].sum()
coverage      = pipe_total / target_next if target_next > 0 else 0

pipe_next_w   = pipe_next.copy()
pipe_next_w['weight']   = pipe_next_w['forecast_category'].map(FORECAST_WEIGHTS).fillna(0.05)
pipe_next_w['weighted'] = pipe_next_w['amount'] * pipe_next_w['weight']
weighted_fcst = pipe_next_w['weighted'].sum()

print(f' Pipeline {NEXT_Q}:')
print(f'   Target:            ${target_next:>10,.0f}')
print(f'   Pipe total:        ${pipe_total:>10,.0f}  ({coverage:.1f}× coverage)')
print(f'   Weighted forecast: ${weighted_fcst:>10,.0f}  ({weighted_fcst/target_next:.0%} of target)')
print(f'   Healthy benchmark: 3× coverage | >85% weighted')

###   Analysing Owner Impact

In [ ]:
# Stall distribution across owners, managers, and active vs inactive reps - If stall concentrates on one manager: coaching problem. If stall is evenly distributed: process problem, not people
# Inactive reps left deals in the funnel that will never progress

stalled_with_meta2 = (stalled
                      .merge(deals_meta[['deal_id','owner_id']], on='deal_id', how='left')
                      .merge(owner[['owner_id','manager','segment']], on='owner_id', how='left'))

print('Stall by owner status (active vs inactive):')
print(stalled.groupby('owner_is_active').agg(
    deals=('deal_id','count'),
    amount=('amount','sum')
).to_string())

print('\nStall by manager:')
print(stalled_with_meta2.groupby('manager').agg(
    deals=('deal_id','count'),
    amount=('amount','sum'),
    avg_weeks=('weeks_in_stage','mean')
).sort_values('deals', ascending=False).to_string())

###   Analysing Deal Properties

In [ ]:
# Stall breakdown by deal source, account size, industry, and deal type - Identifies whether specific channels or account profiles generate disproportionate pipeline that never converts.
# Uniform distribution across channels confirms a process problem, not a lead quality problem from a single source.

stalled_with_meta = (stalled
                     .merge(deals_meta[['deal_id','deal_source','account_size',
                                        'deal_order_type','account_industry']],
                            on='deal_id', how='left'))

print('Stall by deal source:')
print(stalled_with_meta.groupby('deal_source').agg(
    deals=('deal_id','count'),
    amount=('amount','sum'),
    avg_weeks=('weeks_in_stage','mean')
).sort_values('deals', ascending=False).to_string())

print('\nStall by account size:')
print(stalled_with_meta.groupby('account_size').agg(
    deals=('deal_id','count'),
    amount=('amount','sum'),
    avg_weeks=('weeks_in_stage','mean')
).sort_values('deals', ascending=False).to_string())

print('\nStall by deal order type:')
print(stalled_with_meta.groupby('deal_order_type').agg(
    deals=('deal_id','count'),
    amount=('amount','sum'),
    avg_weeks=('weeks_in_stage','mean')
).sort_values('deals', ascending=False).to_string())

## **Statistical rigor** : validating the claims (DMAIC lens)

The sections above establish *what* the numbers are. This section tests *whether the
claims survive scrutiny* — the **Analyze / Control** half of DMAIC:

- **confidence intervals** on win rates (how much can small samples actually claim?),
- a **hypothesis test** behind the "process, not people" conclusion,
- a **data-driven stall threshold** to replace the magic `STALL_WEEKS = 4`,
- **skew-aware** summary stats (mean vs median),
- a **Pareto** for where to act first,
- a **data-quality guardrail** on the Q3 pipeline claim.


In [ ]:
# A. Win rate with 95% Wilson confidence intervals.
# A win rate is a proportion estimated from a sample; the CI shows how much small
# samples (e.g. Cross-Sell, n=44) limit what we can honestly claim.
from scipy.stats import binomtest

def win_ci(won_n, n):
    ci = binomtest(int(won_n), int(n)).proportion_ci(method='wilson')
    return won_n / n, ci.low, ci.high

closed_deals = deals_meta[deals_meta['is_closed']]
rows = [('Overall', int(deals_meta['is_won'].sum()), len(closed_deals))]
for t, g in closed_deals.groupby('deal_order_type'):
    rows.append((t, int(g['is_won'].sum()), len(g)))

win_ci_tbl = pd.DataFrame(
    [(name, w, n, *[f'{v:.1%}' for v in win_ci(w, n)]) for name, w, n in rows],
    columns=['segment', 'won', 'closed', 'win_rate', 'ci_low', 'ci_high'])
print('Win rate - point estimate vs 95% CI (Wilson):')
display(win_ci_tbl)
print('The Cross-Sell / Upsell CIs overlap New Client: the ranking is SUGGESTIVE,\n'
      'not statistically established at these sample sizes.')


In [ ]:
# B. Is stalling associated with WHO owns the deal or WHERE it came from?
# Chi-square test of independence between "stalled" and each categorical factor.
# High p (> 0.05) => no significant association => evidence of a PROCESS problem,
# not a people/channel problem. A permutation p-value is added as a distribution-free
# robustness check.
from scipy.stats import chi2_contingency

open_ids    = set(open_pipeline['deal_id'])
stalled_ids = set(stalled['deal_id'])
stall_base  = (deals_meta[deals_meta['deal_id'].isin(open_ids)]
               [['deal_id', 'deal_source', 'account_size', 'deal_order_type', 'owner_id']]
               .merge(owner[['owner_id', 'manager', 'segment']], on='owner_id', how='left'))
stall_base['stalled'] = stall_base['deal_id'].isin(stalled_ids)

def stall_independence(factor, n_perm=2000, seed=42):
    ct = pd.crosstab(stall_base[factor], stall_base['stalled'])
    chi2, p, dof, exp = chi2_contingency(ct)
    rng = np.random.default_rng(seed)
    labels, cats = stall_base['stalled'].to_numpy(), stall_base[factor].to_numpy()
    perm = np.array([chi2_contingency(pd.crosstab(cats, rng.permutation(labels)))[0]
                     for _ in range(n_perm)])
    return {'factor': factor, 'chi2': round(chi2, 2), 'dof': dof,
            'p_asymptotic': round(p, 3),
            'p_permutation': round(float((perm >= chi2).sum()) / n_perm, 3),
            'min_expected': round(float(exp.min()), 1)}

chi_tbl = pd.DataFrame([stall_independence(f) for f in
                        ['manager', 'deal_source', 'account_size', 'deal_order_type', 'segment']])
print(f'Stall rate among {len(stall_base)} open deals: {stall_base["stalled"].mean():.1%}')
print('H0: stalling is independent of the factor.  p > 0.05 => cannot reject => no association.')
display(chi_tbl)
print('Stalling is independent of manager, source, size and segment (p = 0.30-0.90) -\n'
      'this supports a PROCESS problem over a people/channel one. deal_order_type is\n'
      'borderline (p ~ 0.08): New Client may stall slightly more (a caveat, not a finding).\n'
      'Note: "fail to reject" is not the same as "prove independent".')


In [ ]:
# C. Is a flat 4-week stall threshold defensible? Dwell time differs sharply by stage,
# so one cutoff mislabels deals. Below: per-stage dwell distribution and the p90 upper
# control limit (SPC / Lean Six Sigma) that would replace the magic number.
dwell_by_stage = (spans.groupby('stage')['weeks_in_stage']
                  .agg(n='count', median='median',
                       p90=lambda s: s.quantile(0.90),
                       p95=lambda s: s.quantile(0.95))
                  .reindex(['Prospecting', 'Qualification', 'Proposal', 'Negotiation']))
print(f'Current flat threshold: STALL_WEEKS = {STALL_WEEKS}')
display(dwell_by_stage)
print('4 weeks in Negotiation (median 1) is genuinely stuck; 4 weeks in Prospecting\n'
      "(median 5) is normal. A per-stage control limit (each stage's p90) is the\n"
      'rigorous replacement for a single magic number.')


In [ ]:
# D. Means mislead on skewed distributions - check skew before summarising.
from scipy.stats import skew

def summarise(s):
    s = s.dropna()
    return pd.Series({'mean': s.mean(), 'median': s.median(),
                      'iqr_low': s.quantile(0.25), 'iqr_high': s.quantile(0.75),
                      'skew': round(float(skew(s)), 2)})

skew_tbl = pd.DataFrame({'ASP ($)': summarise(won['deal_amount']),
                         'cycle (days)': summarise(won['cycle_days'])}).T
display(skew_tbl)
print('ASP is right-skewed (skew ~2.6) - report the MEDIAN (~$58.5k), not the mean.\n'
      'Cycle is roughly symmetric (skew ~0.1) - the mean is fine there.')


In [ ]:
# E. Pareto: stall RATE is uniform (above), but stalled VALUE concentrates. This says
# WHERE to start the process fix for the fastest ROI - prioritisation, not blame.
pareto = (stalled.merge(deals_meta[['deal_id', 'owner_id']], on='deal_id')
          .merge(owner[['owner_id', 'manager']], on='owner_id')
          .groupby('manager')['amount'].sum().sort_values(ascending=False)
          .to_frame('stalled_amount'))
pareto['cum_share'] = pareto['stalled_amount'].cumsum() / pareto['stalled_amount'].sum()
n80 = int((pareto['cum_share'] <= 0.80).sum()) + 1
print('Pareto - stalled value by manager (prioritisation, consistent with the uniform rate):')
display(pareto.assign(stalled_amount=lambda d: d['stalled_amount'].map('${:,.0f}'.format),
                      cum_share=lambda d: d['cum_share'].map('{:.0%}'.format)))
print(f'{n80} of {len(pareto)} managers hold ~80% of the stalled value - start there.')


In [ ]:
# F. Data-quality guardrail: is the Q2->Q3 "pipeline collapse" real, or an artifact of
# the export window closing? If snapshot COVERAGE and NEW-deal entry fall together with the
# pipeline in the final weeks, the drop is largely an artifact, not a business signal.
coverage_tail = deals_snapshot.groupby('snapshot_date')['deal_id'].nunique().tail(6)
open_tail = (deals_snapshot[deals_snapshot['stage_order'].isin(OPEN_STAGES)]
             .groupby('snapshot_date')['amount'].sum().tail(6))
entry_weeks = (deals_snapshot.groupby('deal_id')['snapshot_date'].min()
               .dt.to_period('W').value_counts().sort_index())
print(f'Last snapshot in the export: {deals_snapshot["snapshot_date"].max().date()}')
display(pd.DataFrame({'deals_in_snapshot': coverage_tail,
                      'open_pipeline': open_tail.map('${:,.0f}'.format)}))
print(f'Last week any NEW deal entered the pipeline: {entry_weeks.index[-1]}')
print('Coverage falls from ~130 to 95 deals/week and no new deals enter in the final ~3\n'
      'weeks -> the 63% pipeline "drop" is largely a truncated-export ARTIFACT, not a proven\n'
      'decline. The Q3 pipeline signal is not trustworthy near the cutoff (bookings pace is).')


## **Share**

In [ ]:
# Visual 1 — "What blew the forecast?"
COLORS = {'New Client':'#378ADD','Cross-Sell':'#1D9E75','Upsell':'#EF9F27'}

v1 = (won_c
      .groupby(['close_quarter','close_year','close_qnum','deal_order_type'])['deal_amount']
      .sum().reset_index()
      .sort_values(['close_year','close_qnum']))

# Total per quarter for label above each bar
quarter_totals = (won_c
                  .groupby('close_quarter')['deal_amount']
                  .sum()
                  .reset_index()
                  .rename(columns={'deal_amount':'total'}))

bench_total = (won_c[won_c['close_quarter'] != Q1_ANALYSIS]
               .groupby('deal_order_type')['deal_amount']
               .sum()
               .div(won_c[won_c['close_quarter'] != Q1_ANALYSIS]['close_quarter'].nunique())
               .sum())

# Dynamic annotation from Q1_ANALYSIS
q1_mix    = won_c[won_c['close_quarter'] == Q1_ANALYSIS].groupby('deal_order_type')['deal_amount'].sum()
q1_total  = q1_mix.sum()
q1_nc_pct = round(q1_mix.get('New Client', 0) / q1_total * 100)
q1_cs_pct = round(q1_mix.get('Cross-Sell', 0) / q1_total * 100)
q1_up_pct = round(q1_mix.get('Upsell',     0) / q1_total * 100)

fig1 = px.bar(
    v1, x='close_quarter', y='deal_amount', color='deal_order_type',
    color_discrete_map=COLORS,
    category_orders={'close_quarter': QUARTERS_COMPLETOS},
    title=f'{Q1_ANALYSIS}: New Client collapsed −32% YoY — acquisition engine losing steam',
    labels={'deal_amount':'Bookings ($)', 'close_quarter':'', 'deal_order_type':''},
)

# Benchmark line
fig1.add_hline(
    y=bench_total, line_dash='dot', line_color='#E24B4A', line_width=2,
    annotation_text=f'Historical avg ${bench_total/1e3:.0f}k',
    annotation_position='top left',
    annotation_font_color='#E24B4A',
)

# Total label above each bar
for _, row in quarter_totals.iterrows():
    if row['close_quarter'] in QUARTERS_COMPLETOS:
        fig1.add_annotation(
            x=row['close_quarter'],
            y=row['total'],
            text=f"<b>${row['total']/1e3:.0f}k</b>",
            showarrow=False,
            yshift=10,
            font=dict(size=10, color='#333'),
            xanchor='center',
        )

# Q1_ANALYSIS callout
fig1.add_annotation(
    x=Q1_ANALYSIS,
    y=q1_total,
    text=(f'NC {q1_nc_pct}% · CS {q1_cs_pct}% · UP {q1_up_pct}%'),
    showarrow=True, arrowhead=2,
    ax=90, ay=-55,
    font=dict(size=11, color='#E24B4A'),
    bgcolor='white', bordercolor='#E24B4A', borderwidth=1,
)

fig1.update_layout(
    height=480,
    barmode='stack',
    yaxis_tickformat='$,.0f',
    # Legend horizontal below title, full labels visible
    legend=dict(
        orientation='h',
        yanchor='bottom', y=1.04,
        xanchor='center', x=0.5,
        font=dict(size=12),
        title=None,
    ),
    # Extra top margin so labels above bars don't clip
    margin=dict(t=100, r=20, b=40, l=60),
    plot_bgcolor='white',
    paper_bgcolor='white',
)
fig1.show()

In [ ]:
# Map each span to the quarter of its first snapshot
snap_first_q = (deals_snapshot[deals_snapshot['stage_order'].isin(OPEN_STAGES)]
                .sort_values(['deal_id','snapshot_date'])
                .groupby('deal_id')[['snap_quarter','snap_year','snap_qnum']]
                .first()
                .reset_index())

spans_q = (spans
           .merge(snap_first_q, on='deal_id', how='left')
           .query('snap_quarter in @QUARTERS_COMPLETOS'))

# Median weeks per stage × quarter
heat_df = (spans_q
           .groupby(['stage','stage_order','snap_quarter'])['weeks_in_stage']
           .median()
           .reset_index()
           .rename(columns={'weeks_in_stage':'median_weeks'}))

heat_df = heat_df[heat_df['stage'].isin(['Prospecting','Qualification',
                                          'Proposal','Negotiation'])]

heat_pivot = (heat_df
              .pivot(index='stage', columns='snap_quarter', values='median_weeks')
              .reindex(['Prospecting','Qualification','Proposal','Negotiation'])
              .reindex(columns=QUARTERS_COMPLETOS))

# Stall rate annotation per cell
stall_df = (spans_q
            .groupby(['stage','snap_quarter'])['weeks_in_stage']
            .apply(lambda x: (x > 1).mean())
            .reset_index()
            .rename(columns={'weeks_in_stage':'stall_rate'}))

stall_pivot = (stall_df
               .pivot(index='stage', columns='snap_quarter', values='stall_rate')
               .reindex(['Prospecting','Qualification','Proposal','Negotiation'])
               .reindex(columns=QUARTERS_COMPLETOS))

# Build annotation text: "Xw | YY%" per cell
text_matrix = []
for stage in heat_pivot.index:
    row = []
    for q in heat_pivot.columns:
        wk = heat_pivot.loc[stage, q]
        sr = stall_pivot.loc[stage, q]
        if pd.notna(wk) and pd.notna(sr):
            row.append(f'{wk:.0f}w | {sr:.0%}')
        else:
            row.append('')
    text_matrix.append(row)

fig2 = go.Figure(go.Heatmap(
    z=heat_pivot.values,
    x=heat_pivot.columns.tolist(),
    y=heat_pivot.index.tolist(),
    text=text_matrix,
    texttemplate='%{text}',
    textfont=dict(size=11),
    colorscale=[[0,'#9FE1CB'],[0.4,'#EF9F27'],[1,'#E24B4A']],
    colorbar=dict(title='Median<br>weeks'),
    zmin=0,
))

fig2.update_layout(
    title='Stages heatmap — median weeks + stall rate per quarter<br>'
          '<sup>Format: weeks in stage | % stalled (no movement > 1 week)</sup>',
    height=380,
    xaxis=dict(title='Quarter (first snapshot)', side='bottom'),
    yaxis=dict(title='', autorange='reversed'),
    plot_bgcolor='white', paper_bgcolor='white',
)
fig2.show()


In [ ]:
# Visual 3 — Challenge 3: "Are we on track for next quarter?"
# Two panels: Q2 2025 vs Q1 2025 by week & bookings pace vs target

fig3 = make_subplots(
    rows=2, cols=1,
    row_heights=[0.60, 0.40],
    shared_xaxes=False,
    subplot_titles=[
        'Open pipeline by week — Q2 2025 vs Q1 2025 (same week in quarter)',
        'Q2 2025 bookings pace vs target'
    ],
    vertical_spacing=0.14,
)

weeks = comparison['week_in_q'].tolist()

# Q1 reference line
fig3.add_trace(go.Scatter(
    x=weeks,
    y=comparison['pipe_2025 Q1'],
    name='Q1 2025 (reference)',
    mode='lines+markers',
    line=dict(color='#B4B2A9', width=2, dash='dash'),
    marker=dict(size=5),
), row=1, col=1)

# Q2 current line
fig3.add_trace(go.Scatter(
    x=comparison['week_in_q'],
    y=comparison['pipe_2025 Q2'],
    name='Q2 2025',
    mode='lines+markers',
    line=dict(color='#378ADD', width=2.5),
    marker=dict(size=6),
    connectgaps=False,
), row=1, col=1)

# Highlight drop zone — use 'y domain' for subplot-relative yref
fig3.add_shape(
    type='rect',
    x0=6.5, x1=10.5,
    y0=0, y1=1,
    xref='x', yref='y domain',
    fillcolor='#E24B4A', opacity=0.08,
    layer='below', line_width=0,
)

# Annotation on the drop
fig3.add_annotation(
    x=9, y=curr_pipe_q2,
    text=f'<b>{abs(drop_pct):.0f}% drop from peak</b><br>last 4 weeks',
    showarrow=True, arrowhead=2,
    ax=75, ay=-45,
    font=dict(size=11, color='#E24B4A'),
    bgcolor='white', bordercolor='#E24B4A', borderwidth=1,
    row=1, col=1,
)

# Bookings pace bars
fig3.add_trace(go.Bar(
    x=[
        'Target (full Q2)',
        f'Expected by now<br>({pct_elapsed:.0%} elapsed)',
        'Closed so far',
        'Projected run rate',
    ],
    y=[target_q2, target_q2 * pct_elapsed, won_q2_partial, run_rate],
    marker_color=['#B4B2A9', '#EF9F27', '#1D9E75', '#378ADD'],
    text=[
        f'${target_q2/1e6:.2f}M',
        f'${target_q2*pct_elapsed/1e6:.2f}M',
        f'${won_q2_partial/1e6:.2f}M ',
        f'${run_rate/1e6:.2f}M',
    ],
    textposition='outside',
    showlegend=False,
), row=2, col=1)

fig3.update_layout(
    title=(
        f'Q2 2025: bookings {(won_q2_partial/expected_by_now - 1):.0%} ahead of pace '
        f'— but open pipeline dropped {abs(drop_pct):.0f}% in 4 weeks<br>'
        f'<sup>Q3 2025 will inherit a depleted pipeline — immediate rebuild needed</sup>'
    ),
    height=600,
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(t=110, r=20, b=50, l=70),
)

fig3.update_xaxes(title_text='Week in quarter', row=1, col=1, dtick=1)
fig3.update_xaxes(row=2, col=1, tickangle=-15)
fig3.update_yaxes(title_text='Open pipeline ($)', tickformat='$,.2s', row=1, col=1)
fig3.update_yaxes(title_text='Bookings ($)', tickformat='$,.2s', row=2, col=1,
                  range=[0, run_rate * 1.25])

fig3.show()

print(f'\n CRO summary — Q2 2025 ({pct_elapsed:.0%} complete):')
print(f'  Bookings pace:  ${won_q2_partial:,.0f} closed — '
      f'projecting ${run_rate:,.0f} (+{(run_rate/target_q2 - 1):.0%} vs ${target_q2:,.0f} target)')
print(f'  Pipeline drop:  {abs(drop_pct):.0f}% in 4 weeks '
      f'(${peak_pipe_q2:,.0f} → ${curr_pipe_q2:,.0f})')
print(f'  Q3 outlook:     depleted pipeline — needs immediate rebuild')